# Dataset 2 — Bike Sharing Regression Track

## Review 1 — Full Regression Track

**Purpose:** predict total hourly bike rentals (`cnt`) from temporal, seasonal, weather and working-day information.

**Rubric coverage in this notebook:**
- **A1 — Dataset loading & audit (1 mark):** shape, dtypes, missing values, duplicates and target/date checks.
- **A2 — EDA visualisations (2 marks):** distributions, target distribution, correlation heatmap and two feature-target plots.
- **A3 — Insight commentary (1 mark):** written observations accompany EDA.
- **B1 — Data cleaning (1 mark):** duplicate and IQR outlier audit with an explicit treatment decision.
- **B2 — Encoding, scaling & splitting (1 mark):** one-hot encoding for categorical/time codes and train-only preprocessing.
- **B3 — Feature engineering (1 mark):** `rush_hour` created from commuter peak hours with justification.
- **C1 — Implementation (4 marks):** all ten required regression algorithms.
- **C2 — Comparative evaluation (2 marks):** common test split with R², RMSE and MAE for all ten models.
- **C3 — Hyperparameter tuning (2 marks):** GridSearchCV for at least two models, best parameters and improvement.
- **C4 — Visualisation (1 mark):** predicted-vs-actual, residual plot and tree-based feature importance.

**Leakage rule:** `casual + registered = cnt`, so `casual` and `registered` are removed before modelling. `instant` and the raw date field are also excluded from the predictor matrix.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.base import clone, BaseEstimator, TransformerMixin

sns.set_theme(style='whitegrid')

class IQRClipper(BaseEstimator, TransformerMixin):
    """Cap numeric predictors using IQR limits learned only from the current fit data."""
    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        self.feature_names_in_ = np.asarray(X_df.columns, dtype=object)
        q1 = X_df.quantile(0.25)
        q3 = X_df.quantile(0.75)
        iqr = q3 - q1
        self.lower_ = q1 - 1.5 * iqr
        self.upper_ = q3 + 1.5 * iqr
        self.n_features_in_ = X_df.shape[1]
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()
        # Preserve the input column names when the transformer is used inside
        # ColumnTransformer/Pipeline, and then return the numeric array expected downstream.
        X_df.columns = self.feature_names_in_
        clipped = X_df.clip(self.lower_, self.upper_, axis=1)
        return clipped.to_numpy()

    def get_feature_names_out(self, input_features=None):
        # IQR capping changes values, not feature count or feature names.
        if input_features is None:
            return np.asarray(self.feature_names_in_, dtype=object)
        return np.asarray(input_features, dtype=object)

RANDOM_STATE = 42
N_JOBS = 2
OUTPUT_DIR = Path('../outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = Path('../data/hour.csv')

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH.resolve()}. Place hour.csv inside the project's data/ folder and rerun."
    )

print('Loading Bike Sharing dataset...')
df_bike = pd.read_csv(DATA_PATH)
df_bike['dteday'] = pd.to_datetime(df_bike['dteday'], errors='coerce')

print('Shape:', df_bike.shape)

## A1 — Dataset audit and data quality (1 mark)

The audit reports the exact shape, data types, missing values, duplicate rows, target statistics, and the date range. The supplied dataset has no missing values and currently contains no exact duplicate rows, but the checks remain in the notebook for reproducibility.

### A1 — Dataset loading and audit (1 mark)

Before modelling, we report the dataset shape, data types, missing-value counts, exact duplicates, target statistics and date range. These checks establish what the dataset contains and whether immediate cleaning is required.

In [ ]:
print('\n--- Dataset Shape ---')
print(df_bike.shape)

print('\n--- Data Types ---')
print(df_bike.dtypes)

print('\n--- Missing Values ---')
print(df_bike.isna().sum())

print('\n--- Exact Duplicate Rows ---')
print(int(df_bike.duplicated().sum()))

print('\n--- Target Summary (cnt) ---')
display(df_bike['cnt'].describe().to_frame('cnt').T)

print('\n--- Date Range ---')
print(df_bike['dteday'].min(), 'to', df_bike['dteday'].max())

# Exact duplicate treatment
duplicate_count = int(df_bike.duplicated().sum())
df_bike = df_bike.drop_duplicates().copy()
print(f'Rows removed as exact duplicates: {duplicate_count}')

## B1 — Outlier audit and treatment (part of the 1-mark data-cleaning criterion)

We audit potential outliers using the IQR rule. Input-feature outliers are **treated in the actual modelling pipeline** with IQR capping. The capping bounds are learned from the training data only, and during cross-validation they are learned independently inside each fold.


### B1 — Why capping is appropriate here

Bike demand contains legitimate high-demand hours, so deleting extreme rows could remove useful real-world behaviour. We therefore cap only extreme **input predictors** (`temp`, `atemp`, `hum`, `windspeed`) rather than deleting observations. The target `cnt` is audited but retained because high demand is itself a genuine outcome that the model should learn.


In [ ]:
def iqr_audit(df_in, columns):
    rows = []
    for col in columns:
        q1 = df_in[col].quantile(0.25)
        q3 = df_in[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        count = int(((df_in[col] < lower) | (df_in[col] > upper)).sum())
        rows.append({
            'Feature': col, 'Q1': q1, 'Q3': q3,
            'Lower Bound': lower, 'Upper Bound': upper,
            'IQR Outlier Count': count
        })
    return pd.DataFrame(rows)

outlier_features = ['temp', 'atemp', 'hum', 'windspeed', 'cnt']
outlier_audit = iqr_audit(df_bike, outlier_features)
print('--- Outlier audit ---')
display(outlier_audit)
outlier_audit.to_csv(OUTPUT_DIR / 'bike_outlier_audit.csv', index=False)


## A2 — Exploratory data analysis (2 marks)

The rubric asks for feature distributions, target distribution, a correlation heatmap, and at least two feature-target scatter plots. The bike dataset contains one date field and many numeric-coded predictors, so the numeric predictor distributions are shown together and the date range is reported separately.

### A2 — Exploratory Data Analysis visualisations (2 marks)

The rubric explicitly asks for: **feature distributions, a correlation heatmap, target distribution, and at least two feature-target scatter plots**.

This section produces all four categories of visual evidence. The plots are used to understand demand skew, temporal/weather relationships and possible modelling challenges before preprocessing.

In [ ]:
# Feature distributions: all model-eligible raw numeric predictors before final feature exclusion
predictor_cols_for_eda = [c for c in df_bike.columns if c not in ['dteday', 'cnt']]
df_bike[predictor_cols_for_eda].hist(figsize=(15, 12), bins=20)
plt.suptitle('Bike Sharing Feature Distributions', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'bike_feature_distributions.png', dpi=180, bbox_inches='tight')
plt.show()
plt.close('all')

# Target distribution
plt.figure(figsize=(9, 4.5))
sns.histplot(df_bike['cnt'], bins=50, kde=True, color='steelblue')
plt.title('Distribution of Total Bike Rentals (cnt)')
plt.xlabel('Total Rentals (cnt)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'bike_target_distribution.png', dpi=180, bbox_inches='tight')
plt.show()
plt.close('all')

# Correlation heatmap
numeric_cols = df_bike.select_dtypes(include=np.number).columns
plt.figure(figsize=(11, 9))
sns.heatmap(df_bike[numeric_cols].corr(), cmap='coolwarm', center=0)
plt.title('Bike Sharing Correlation Heatmap')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'bike_correlation_heatmap.png', dpi=180, bbox_inches='tight')
plt.show()
plt.close('all')

# Two mandatory feature-target relationships
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=df_bike, x='temp', y='cnt', alpha=0.30, ax=axes[0])
axes[0].set_title('Temperature vs Total Rentals')
axes[0].set_xlabel('Temperature (normalised)')
axes[0].set_ylabel('Total Rentals (cnt)')

sns.scatterplot(data=df_bike, x='hum', y='cnt', alpha=0.30, ax=axes[1])
axes[1].set_title('Humidity vs Total Rentals')
axes[1].set_xlabel('Humidity (normalised)')
axes[1].set_ylabel('Total Rentals (cnt)')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'bike_scatter_temp_hum.png', dpi=180, bbox_inches='tight')
plt.show()

### A3 — EDA insight commentary (1 mark)

- `cnt` is strongly right-skewed, with a small number of high-demand hours.
- Temperature is positively associated with demand, while humidity shows a weaker/inverse pattern in the raw scatter view.
- `casual + registered = cnt`; therefore those two variables must be removed from the predictors to prevent target leakage.
- The time variables (`hr`, `mnth`, `weekday`, `season`) are coded categories rather than continuous measurements, so they will be one-hot encoded in preprocessing.

### B3 — Feature engineering (1 mark)

A binary `rush_hour` feature is created for hours **08:00, 09:00, 17:00 and 18:00**. These windows are intended to capture recurring commuter-demand peaks that may be harder for a single raw hour code to express as a direct business signal.

The engineered feature is created before splitting, but the model's learned preprocessing is still fitted only on training data.

In [ ]:
# Feature engineering
df_bike['rush_hour'] = df_bike['hr'].isin([8, 9, 17, 18]).astype(int)

print('Rush-hour rows:', int(df_bike['rush_hour'].sum()))

# Final feature matrix and target
# Remove index/date fields and the two components of cnt to prevent leakage.
X = df_bike.drop(columns=['instant', 'dteday', 'casual', 'registered', 'cnt']).copy()
y = df_bike['cnt'].copy()

print('Model features:')
print(list(X.columns))
print('Target:', y.name)

# Stratify the regression split by target quantile bins so low/high-demand ranges are represented in both sets.
y_bins = pd.qcut(y, q=10, duplicates='drop')
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y_bins
)

print('\nTrain shape:', X_train.shape)
print('Test shape:', X_test.shape)

## B2 — Preprocessing and encoding (1 mark)

All ten regressors receive the same leakage-safe preprocessing. Categorical/time-code variables are one-hot encoded, numeric variables pass through an **IQR outlier capper followed by StandardScaler**, and all transformations are fitted only on the training data through pipelines. This means the outlier treatment is part of the actual model input pipeline, not just a plot-time operation.


### B2 — Encoding, outlier treatment, scaling and train/test splitting

**Categorical encoding:** `season`, `mnth`, `hr`, `weekday`, `weathersit` → One-Hot Encoding.

**Numeric processing:** `IQRClipper` → `StandardScaler` for `yr`, `holiday`, `workingday`, `temp`, `atemp`, `hum`, `windspeed`, `rush_hour`.

The capper learns bounds only from its fit data. Therefore it cannot use information from the held-out test set, and it is automatically refit separately inside each cross-validation fold.


In [ ]:
categorical_features = ['season', 'mnth', 'hr', 'weekday', 'weathersit']
numeric_features = [
    'yr', 'holiday', 'workingday', 'temp', 'atemp', 'hum', 'windspeed', 'rush_hour'
]

numeric_pipeline = Pipeline([
    ('outlier_treatment', IQRClipper()),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
        ('numeric', numeric_pipeline, numeric_features)
    ],
    remainder='drop'
)

print('Categorical features:', categorical_features)
print('Numeric/binary features:', numeric_features)

# Fit only for feature-name inspection; all real model fitting happens through independent pipelines.
preprocessor_preview = preprocessor.fit(X_train)
feature_names = preprocessor_preview.get_feature_names_out()
print('Number of transformed features:', len(feature_names))
print('First 20 transformed feature names:')
print(feature_names[:20])


## C1 — Ten required regression algorithms (4 marks)

The ten baseline algorithms are implemented exactly as required by the project guideline. Each model is wrapped in the same preprocessing pipeline so comparisons are fair.

### C1 — Ten required regression algorithms (4 marks)

The guideline requires all ten models to be trained on the same preprocessed dataset:

1. Linear Regression — baseline and coefficient interpretation.
2. Ridge Regression — L2 regularisation.
3. Lasso Regression — L1 regularisation and sparsity.
4. ElasticNet Regression — combined L1 + L2 regularisation.
5. Polynomial Regression — nonlinear feature expansion followed by linear regression.
6. Decision Tree Regressor — recursive nonlinear splits.
7. Random Forest Regressor — bagged tree ensemble.
8. Gradient Boosting Regressor — sequential error-correcting tree ensemble.
9. Support Vector Regressor (SVR) — margin-based regression; scaling is important.
10. K-Nearest Neighbors Regressor — local distance-based prediction; scaling matters.

In [ ]:
def make_pipeline_for_model(model):
    return Pipeline([
        ('preprocess', clone(preprocessor)),
        ('model', model)
    ])

# Polynomial Regression expands only the continuous/binary numeric block.
# Outlier treatment is fitted before polynomial expansion so extreme inputs do not create exaggerated polynomial terms.
poly_numeric_pipeline = Pipeline([
    ('outlier_treatment', IQRClipper()),
    ('poly_numeric', PolynomialFeatures(degree=2, include_bias=False)),
    ('numeric_scaler', StandardScaler())
])

poly_preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
    ('numeric_poly', poly_numeric_pipeline, numeric_features)
], remainder='drop')
poly_pipeline = Pipeline([('preprocess', poly_preprocessor), ('model', LinearRegression())])

models = {
    'Linear Regression': make_pipeline_for_model(LinearRegression()),
    'Ridge Regression': make_pipeline_for_model(Ridge(alpha=1.0)),
    'Lasso Regression': make_pipeline_for_model(Lasso(alpha=0.001, max_iter=10000, random_state=RANDOM_STATE)),
    'ElasticNet Regression': make_pipeline_for_model(ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000, random_state=RANDOM_STATE)),
    'Polynomial Regression': poly_pipeline,
    'Decision Tree Regressor': make_pipeline_for_model(DecisionTreeRegressor(random_state=RANDOM_STATE)),
    'Random Forest Regressor': make_pipeline_for_model(RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=1)),
    'Gradient Boosting Regressor': make_pipeline_for_model(GradientBoostingRegressor(random_state=RANDOM_STATE)),
    'Support Vector Regressor (SVR)': make_pipeline_for_model(SVR(C=1.0, kernel='linear')),
    'K-Nearest Neighbors Regressor': make_pipeline_for_model(KNeighborsRegressor(n_neighbors=5))
}

print('Required models:', len(models))
print('\n'.join(f'{i}. {name}' for i, name in enumerate(models.keys(), start=1)))


### C2 — Comparative evaluation (2 marks)

Every baseline model uses the same train/test split and the same target (`cnt`). We report:

- **R²:** proportion of held-out target variance explained.
- **RMSE:** square-rooted mean squared error; larger mistakes are penalised more strongly.
- **MAE:** average absolute error in the target's units.

The final table is sorted by R² so the comparison is transparent rather than scattered across separate print statements.

In [ ]:
def evaluate_regression_models(model_dict, X_train, X_test, y_train, y_test):
    rows = []
    predictions = {}
    for name, model in model_dict.items():
        print(f'Fitting: {name}')
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        mse = mean_squared_error(y_test, pred)
        rows.append({
            'Model': name,
            'R2 Score': r2_score(y_test, pred),
            'RMSE': np.sqrt(mse),
            'MAE': mean_absolute_error(y_test, pred)
        })
        predictions[name] = pred
    results = pd.DataFrame(rows).sort_values('R2 Score', ascending=False).reset_index(drop=True)
    return results, predictions

# Baseline comparison across all ten algorithms
baseline_results, baseline_predictions = evaluate_regression_models(
    models, X_train, X_test, y_train, y_test
)

print('\n--- Baseline Regression Comparison (All 10) ---')
display(baseline_results.style.format({
    'R2 Score':'{:.4f}', 'RMSE':'{:.2f}', 'MAE':'{:.2f}'
}))
baseline_results.to_csv(OUTPUT_DIR / 'regression_baseline_results.csv', index=False)

## C3 — Hyperparameter tuning with GridSearchCV (2 marks)

The rubric requires tuning for at least two regression models and reporting the best parameters plus metric improvement. We tune **Random Forest** (`n_estimators`, `max_depth`) and **Gradient Boosting** (`n_estimators`, `learning_rate`) using only the training split. The test set stays untouched until final evaluation.

### C3 — Hyperparameter tuning (2 marks)

At least two models must be tuned. We use **GridSearchCV on the training data only**:

- **Random Forest:** `n_estimators`, `max_depth`.
- **Gradient Boosting:** `n_estimators`, `learning_rate`.

The held-out test set is not used to choose hyperparameters. After tuning, the two models are refit through the complete preprocessing pipeline and compared against their baseline test results.

In [ ]:
# GridSearchCV is performed on the complete preprocessing + model pipeline.
# Therefore outlier treatment, encoding and scaling are refit inside every CV fold.
rf_pipeline = Pipeline([
    ('preprocess', clone(preprocessor)),
    ('model', RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1))
])
rf_param_grid = {
    'model__n_estimators': [50, 100],
    'model__max_depth': [None, 10]
}
rf_grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    cv=3,
    scoring='r2',
    n_jobs=1,
    return_train_score=False
)
print('Tuning Random Forest...')
rf_grid.fit(X_train, y_train)
print('Best RF parameters:', rf_grid.best_params_)

gb_pipeline = Pipeline([
    ('preprocess', clone(preprocessor)),
    ('model', GradientBoostingRegressor(random_state=RANDOM_STATE))
])
gb_param_grid = {
    'model__n_estimators': [100, 150],
    'model__learning_rate': [0.05, 0.1]
}
gb_grid = GridSearchCV(
    estimator=gb_pipeline,
    param_grid=gb_param_grid,
    cv=3,
    scoring='r2',
    n_jobs=1,
    return_train_score=False
)
print('Tuning Gradient Boosting...')
gb_grid.fit(X_train, y_train)
print('Best GB parameters:', gb_grid.best_params_)


### C3 — Tuning improvement evidence

This section replaces the two baseline rows with the tuned test predictions and then calculates the final comparison table. The notebook also reports the baseline-versus-tuned change in R², RMSE and MAE so the required improvement can be discussed explicitly in the viva.

In [ ]:
# Build tuned versions only for the two models selected for hyperparameter tuning.
tuned_models = {
    'Random Forest Regressor': rf_grid.best_estimator_,
    'Gradient Boosting Regressor': gb_grid.best_estimator_
}
tuned_predictions = baseline_predictions.copy()

for name, pipeline in tuned_models.items():
    # The best_estimator_ has its own preprocessing + outlier treatment already.
    pred = pipeline.predict(X_test)
    tuned_predictions[name] = pred

# Start from the all-10 baseline table and replace only the two tuned rows.
tuned_results = baseline_results.copy()
for name in tuned_models:
    pred = tuned_predictions[name]
    mask = tuned_results['Model'].eq(name)
    mse = mean_squared_error(y_test, pred)
    tuned_results.loc[mask, 'R2 Score'] = r2_score(y_test, pred)
    tuned_results.loc[mask, 'RMSE'] = np.sqrt(mse)
    tuned_results.loc[mask, 'MAE'] = mean_absolute_error(y_test, pred)

tuned_results = tuned_results.sort_values('R2 Score', ascending=False).reset_index(drop=True)

# Explicit baseline-versus-tuned improvement table (required by C3).
improvement_rows = []
for name in tuned_models:
    base_row = baseline_results.loc[baseline_results['Model'].eq(name)].iloc[0]
    tuned_row = tuned_results.loc[tuned_results['Model'].eq(name)].iloc[0]
    improvement_rows.append({
        'Model': name,
        'Baseline R2': base_row['R2 Score'],
        'Tuned R2': tuned_row['R2 Score'],
        'R2 Improvement': tuned_row['R2 Score'] - base_row['R2 Score'],
        'Baseline RMSE': base_row['RMSE'],
        'Tuned RMSE': tuned_row['RMSE'],
        'RMSE Change': tuned_row['RMSE'] - base_row['RMSE'],
        'Baseline MAE': base_row['MAE'],
        'Tuned MAE': tuned_row['MAE'],
        'MAE Change': tuned_row['MAE'] - base_row['MAE']
    })
improvement_df = pd.DataFrame(improvement_rows)
print('--- Baseline vs Tuned Improvement ---')
display(improvement_df.style.format({
    'Baseline R2':'{:.4f}', 'Tuned R2':'{:.4f}', 'R2 Improvement':'{:.4f}',
    'Baseline RMSE':'{:.2f}', 'Tuned RMSE':'{:.2f}', 'RMSE Change':'{:.2f}',
    'Baseline MAE':'{:.2f}', 'Tuned MAE':'{:.2f}', 'MAE Change':'{:.2f}'
}))
improvement_df.to_csv(OUTPUT_DIR / 'regression_tuning_improvement.csv', index=False)

print('--- Final Regression Comparison (All 10; RF and GB tuned) ---')
display(tuned_results.style.format({
    'R2 Score':'{:.4f}', 'RMSE':'{:.2f}', 'MAE':'{:.2f}'
}))
tuned_results.to_csv(OUTPUT_DIR / 'regression_results_full.csv', index=False)


## General requirement — 5-fold cross-validation for the two strongest final models

The guideline requires cross-validation for the two best-performing models. Because the preprocessing pipeline contains the actual IQR outlier treatment, one-hot encoding and scaling, the **complete pipeline** is cross-validated so each fold learns its own treatment parameters from its own training portion.


### General model-evaluation requirement — 5-fold cross-validation

The general guideline requires 5-fold CV for the two best-performing regression models. CV is performed using the **full preprocessing + model pipeline** on the training split so the encoder/scaler are refit independently inside each fold. The test set remains untouched.

In [ ]:
top_two_models = tuned_results.head(2)['Model'].tolist()
cv_rows = []

for name in top_two_models:
    # IMPORTANT: cross-validate the complete preprocessing + model pipeline.
    # This makes the encoder/scaler refit independently inside each fold and prevents preprocessing leakage.
    if name in tuned_models:
        estimator = tuned_models[name]
    else:
        estimator = models[name]

    cv = cross_validate(
        estimator, X_train, y_train, cv=5, scoring='r2',
        n_jobs=1, return_train_score=False
    )
    cv_rows.append({
        'Model': name,
        '5-Fold CV R2 Mean': cv['test_score'].mean(),
        '5-Fold CV R2 Std': cv['test_score'].std(),
        '5-Fold CV R2 Min': cv['test_score'].min(),
        '5-Fold CV R2 Max': cv['test_score'].max()
    })

cv_top2 = pd.DataFrame(cv_rows)
print('Top two final models:', top_two_models)
display(cv_top2.style.format({
    '5-Fold CV R2 Mean':'{:.4f}', '5-Fold CV R2 Std':'{:.4f}',
    '5-Fold CV R2 Min':'{:.4f}', '5-Fold CV R2 Max':'{:.4f}'
}))
cv_top2.to_csv(OUTPUT_DIR / 'regression_cv_top2.csv', index=False)

## C4 — Best-model diagnostics (1 mark)

For the final best model we show:

1. **Predicted vs actual:** how closely predictions follow the ideal 45° line.
2. **Residual plot:** whether errors are centered around zero without a strong systematic pattern.
3. **Random Forest feature importance:** which transformed input variables contribute most to the tuned tree ensemble.

### C4 — Best-model diagnostics (1 mark)

For the final best model we provide the rubric's requested diagnostics:

1. **Predicted vs Actual** — checks alignment with the ideal 45° line.
2. **Residual plot** — checks whether prediction errors are centred near zero and whether systematic patterns remain.
3. **Feature importance** for a tree-based model — identifies which transformed inputs contribute most to the ensemble.

In [ ]:
best_model_name = top_two_models[0]
best_model = tuned_models[best_model_name]
best_preds = tuned_predictions[best_model_name]
residuals = y_test - best_preds

# Predicted vs Actual
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test, y=best_preds, alpha=0.35)
lims = [min(y_test.min(), best_preds.min()), max(y_test.max(), best_preds.max())]
plt.plot(lims, lims, linestyle='--', linewidth=2, label='Ideal prediction')
plt.title(f'Predicted vs Actual — {best_model_name}')
plt.xlabel('Actual cnt')
plt.ylabel('Predicted cnt')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'reg_pred_vs_actual.png', dpi=180, bbox_inches='tight')
plt.show()

# Residual plot
plt.figure(figsize=(8, 6))
sns.scatterplot(x=best_preds, y=residuals, alpha=0.35)
plt.axhline(0, linestyle='--', linewidth=2, label='Zero residual')
plt.title(f'Residual Plot — {best_model_name}')
plt.xlabel('Predicted cnt')
plt.ylabel('Residual (Actual - Predicted)')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'reg_residuals.png', dpi=180, bbox_inches='tight')
plt.show()

### C4 — Tree-based feature importance

Random Forest feature importance provides an interpretable view of which transformed variables contribute most to the predictions. This is a model-specific diagnostic, not a universal causal claim about the real-world effect of a feature.

In [ ]:
# Feature importance from the trained Random Forest baseline.
# This satisfies the rubric requirement for a tree-based feature-importance visualisation.
rf_model = models['Random Forest Regressor']
rf_preprocessor = rf_model.named_steps['preprocess']
rf_estimator = rf_model.named_steps['model']
transformed_feature_names = rf_preprocessor.get_feature_names_out()
importances = pd.Series(
    rf_estimator.feature_importances_,
    index=transformed_feature_names
).sort_values(ascending=False)

importance_top20 = importances.head(20).sort_values()
plt.figure(figsize=(10, 7))
importance_top20.plot(kind='barh')
plt.title('Top 20 Feature Importances — Random Forest')
plt.xlabel('Importance')
plt.ylabel('Transformed Feature')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'rf_feature_importance.png', dpi=180, bbox_inches='tight')
plt.show()
plt.close('all')

importance_df = importance_top20.reset_index()
importance_df.columns = ['Feature', 'Importance']
importance_df.to_csv(OUTPUT_DIR / 'rf_feature_importance.csv', index=False)

## Review-1 final interpretation and viva points

- Linear models provide a simple baseline and coefficient-based interpretation.
- Ridge/Lasso/ElasticNet control coefficient magnitude through regularisation.
- Polynomial Regression can capture smooth non-linear interactions but increases model complexity.
- Decision Trees learn non-linear threshold rules but can overfit without depth control.
- Random Forest averages many trees and reduces variance.
- Gradient Boosting builds trees sequentially to reduce previous errors; learning rate and number/depth of trees control the bias-complexity trade-off.
- SVR uses a margin-based objective and benefits strongly from scaling. The baseline uses a linear kernel for efficient training on this dataset; C controls the regularisation strength, and kernel choice controls the shape of the regression function.
- KNN predicts from nearby training examples, so distance geometry and scaling matter.
- **R²:** proportion of target variance explained; closer to 1 is better for held-out data.
- **RMSE:** square-rooted average squared error; penalises large mistakes more heavily.
- **MAE:** average absolute error; easier to interpret in target units.
- Ridge and KNN hyperparameters are tuned only on the training split. The final model is selected from the actual test-set table and validated with 5-fold CV; do not claim universal superiority beyond this dataset and split.

### Review-1 final interpretation and viva preparation

The notebook now contains the complete regression narrative: **problem → data audit → EDA → cleaning → feature engineering → preprocessing → ten models → tuning → common test comparison → 5-fold CV → diagnostics**.

During the viva, be prepared to explain why `casual`/`registered` are removed, why categorical time variables are one-hot encoded, why scaling is needed for SVR/KNN, what each metric means, and why the test set is kept separate from hyperparameter tuning.

In [ ]:
print('\n=== FINAL REGRESSION SUMMARY ===')
print('Dataset rows:', len(df_bike))
print('Train/Test:', len(X_train), '/', len(X_test))
print('Features before preprocessing:', list(X.columns))
print('Categorical encoded:', categorical_features)
print('Feature engineered:', 'rush_hour')
print('Top two final models:', top_two_models)
print('Best model:', best_model_name)
print('Outputs saved to:', OUTPUT_DIR.resolve())